In [2]:
from __future__ import annotations
from dataclasses import dataclass, field
from collections import defaultdict
from collections import deque
from typing import TypeVar, Generic, Dict, List, Optional, Tuple, Set

In [3]:
@dataclass(eq=True, frozen=True)
class Node:
    name: str

    def __repr__(self):
        return f"Node({self.name!r})"

Graph = Dict[Node, set[Node]]
def build_graph(edges: List[Tuple[str, str]]) -> Graph:
    nodes: Dict[str, Node] = {}
    graph: Graph = defaultdict(set)
    for src, dst in edges:
        if src not in nodes:
            nodes[src] = Node(src)
        if dst not in nodes:
            nodes[dst] = Node(dst)
        graph[nodes[src]].add(nodes[dst])
        graph[nodes[dst]].add(nodes[src])
    return graph


In [4]:
build_graph([("A", "B"), ("B", "C"), ("C", "D"), ("D", "A")])

defaultdict(set,
            {Node('A'): {Node('B'), Node('D')},
             Node('B'): {Node('A'), Node('C')},
             Node('C'): {Node('B'), Node('D')},
             Node('D'): {Node('A'), Node('C')}})

In [5]:
def bfs(graph: Graph, start: Node) -> List[Node]:
    visited: Set[Node] = set()
    queue: deque[Node] = deque([start])
    visited.add(start)
    result: List[Node] = []

    while queue:
        current = queue.popleft()
        result.append(current)
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return result

In [6]:
bfs(build_graph([("A", "B"), ("B", "A")]), Node("A"))

[Node('A'), Node('B')]

#### BFS: with external visited set

In [7]:

def bfs(graph: Graph, start: Node, visited: set[Node] = None) -> List[Node]:
    if visited is None:
        visited = set()
    if start in visited:
        return []
    visited.add(start)
    queue: deque[Node] = deque([start])
    result: List[Node] = []
    while queue:
        current = queue.popleft()
        result.append(current)
        for neighbor in graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return result

In [8]:
bfs(build_graph([("A", "B"), ("B", "A")]), Node("A"))

[Node('A'), Node('B')]

In [9]:
def connected_components(graph: Graph) -> list[set[None]]:
    visited: set[Node] = set()
    res: list[set[Node]] = []
    for v in graph:
        if c:=set(bfs(graph, v, visited)):
            res.append(c)
    return res




In [10]:
connected_components(build_graph([("A", "B"), ("B", "C"), ("AA", "BB")]))

[{Node('A'), Node('B'), Node('C')}, {Node('AA'), Node('BB')}]

In [11]:
@dataclass
class Node:
    name: str
    neighbours: list["Node"] = field(default_factory=list)

    def __str__(self) -> str:
        return self.name

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Node):
            return NotImplemented
        return self.name == other.name and set(self.neighbours) == set(other.neighbours)

    def __hash__(self) -> int:
        return hash((self.name, tuple(sorted(self.neighbours, key=lambda n: n.name))))
    
    def __repr__(self) -> str:
        return f"Node(name={self.name}, neighbours={[n.name for n in self.neighbours]})"


# Node("start", [Node("A"), Node("B")]) == Node("start", [Node("B"), Node("A")])  # True
# Node("start", [Node("A"), Node("B")])
# Node("end")

@dataclass
class Graph:
    nodes: dict[str, Node] = field(default_factory=lambda: defaultdict(Node))

    def add_edge(self, from_node: str, to_node: str) -> None:
        if from_node not in self.nodes:
            self.nodes[from_node] = Node(from_node)
        if to_node not in self.nodes:
            self.nodes[to_node] = Node(to_node)
        self.nodes[from_node].neighbours.append(self.nodes[to_node])
        self.nodes[to_node].neighbours.append(self.nodes[from_node])
    
    def __repr__(self) -> str:
        return f"Graph({list(self.nodes.values())})"


In [12]:
Graph()

Graph([])

In [13]:
import inspect
def get_class_methods(cls):
    return [name for name, method in inspect.getmembers(cls, predicate=inspect.isfunction)]

# Example usage of get_class_methods
print(get_class_methods(Node))  # Should print the names of methods defined in Node class


['__eq__', '__hash__', '__init__', '__replace__', '__repr__', '__str__']


In [14]:
print(inspect.getsource(Node.__eq__))

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Node):
            return NotImplemented
        return self.name == other.name and set(self.neighbours) == set(other.neighbours)



In [15]:
@dataclass
class Graph:
    nodes: list[Node]

In [16]:
@dataclass(frozen=True)
class Node:
    name: str

    def __str__(self) -> str:
        return self.name


@dataclass
class Graph:
    vertices: set[Node]
    adjacency_list: dict[Node, list[Node]]

    def __init__(self):
        self.vertices = set()
        self.adjacency_list = defaultdict(list)

    def add_vertex(self, v: Node) -> None:
        if v not in self.vertices:
            self.vertices.add(v)
            self.adjacency_list[v] = []

    def get_neighbours(self, v: Node) -> list[Node]:
        return self.adjacency_list[v] if v in self.adjacency_list else []

    def add_edge(self, u: Node, v: Node) -> None:
        if u not in self.vertices:
            self.add_vertex(u)
        if v not in self.vertices:
            self.add_vertex(v)
        self.adjacency_list[u].append(v)
        # self.edges[v].append(u)

    def __repr__(self):
        return "\n".join(
            [
                f"{v} -> {[str(x) for x in self.adjacency_list[v]]}"
                for v in self.vertices
            ]
        )

In [17]:
g = Graph()
g.add_vertex(Node("A"))
g.add_edge(Node("A"), Node("B"))
g.add_edge(Node("A"), Node("C"))
g.add_edge(Node("B"), Node("D"))
g.add_edge(Node("C"), Node("F"))
g.add_edge(Node("B"), Node("E"))
g.add_edge(Node("E"), Node("F"))

In [18]:
g

D -> []
F -> []
B -> ['D', 'E']
E -> ['F']
C -> ['F']
A -> ['B', 'C']

In [19]:
[
    [1, 1, 1, 0, 0, 0],
    [0, 1, 0, 1, 1, 0],
    [0, 0, 1, 0, 0, 1],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 1, 1],
    [0, 0, 0, 0, 0, 1],
]

[[1, 1, 1, 0, 0, 0],
 [0, 1, 0, 1, 1, 0],
 [0, 0, 1, 0, 0, 1],
 [0, 0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1, 1],
 [0, 0, 0, 0, 0, 1]]

In [20]:
queue: deque[Node] = deque()


def bfs(g: Graph, start: Node):
    visited: set[Node] = set()
    queue.append(start)
    visited.add(start)
    count = 0
    path: list[Node] = []
    while queue:
        v = queue.popleft()

        path.append(v)
        count += 100
        for u in g.get_neighbours(v):
            if u not in visited:
                queue.append(u)
                visited.add(u)

    return f"{path}: {count}"


def dfs(g: Graph, start: Node):
    visited: set[Node] = set()
    stack = [start]
    visited.add(start)
    while stack:
        v = stack.pop()
        print(v, end=" ")
        for u in g.get_neighbours(v):
            if u not in visited:
                stack.append(u)
                visited.add(u)

In [21]:
bfs(g, Node("B"))

"[Node(name='B'), Node(name='D'), Node(name='E'), Node(name='F')]: 400"

In [22]:
for v in g.vertices:
    print(v, bfs(g, v))

D [Node(name='D')]: 100
F [Node(name='F')]: 100
B [Node(name='B'), Node(name='D'), Node(name='E'), Node(name='F')]: 400
E [Node(name='E'), Node(name='F')]: 200
C [Node(name='C'), Node(name='F')]: 200
A [Node(name='A'), Node(name='B'), Node(name='C'), Node(name='D'), Node(name='E'), Node(name='F')]: 600
